In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("../historico/historical_assets.csv")

df['date'] = pd.to_datetime(df['date'])

dfs_por_ticker = {tick: df_tick for tick, df_tick in df.groupby("ticker")}


df_appl = dfs_por_ticker["AAPL"]

ts = df_appl.set_index('date')['Close']

print(ts)

date
1980-12-12      0.128348
1980-12-15      0.121652
1980-12-16      0.112723
1980-12-17      0.115513
1980-12-18      0.118862
                 ...    
2026-02-24    272.140015
2026-02-25    274.230011
2026-02-26    272.950012
2026-02-27    264.179993
2026-03-20    247.990005
Name: Close, Length: 11395, dtype: float64


In [9]:
train = ts[:-10]  # últimos 10 días como test
test = ts[-10:]

In [10]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Parámetros iniciales: (p,d,q)(P,D,Q,s)
# Supongamos ciclo semanal de 5 días
model = SARIMAX(train,
                order=(1,1,1),
                seasonal_order=(1,1,1,5),
                enforce_stationarity=False,
                enforce_invertibility=False)

model_fit = model.fit(disp=False)
print(model_fit.summary())

c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


                                     SARIMAX Results                                     
Dep. Variable:                             Close   No. Observations:                11385
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 5)   Log Likelihood              -17745.140
Date:                           Mon, 23 Mar 2026   AIC                          35500.281
Time:                                   10:26:08   BIC                          35536.975
Sample:                                        0   HQIC                         35512.623
                                         - 11385                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1645      0.093      1.764      0.078      -0.018       0.347
ma.L1         -0.1384      0.094     -1.478

In [14]:
# Predicción de 1 paso
pred_today = model_fit.get_forecast(steps=1).predicted_mean.iloc[0]

print(f"Predicción precio de cierre hoy: {pred_today:.2f}")


Predicción precio de cierre hoy: 255.68


c:\Users\jorge\miniconda3\envs\proyecto_env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
